[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c04_limits.ipynb)

# The ghosts that nearly broke calculus: limits and the epsilon-delta game

In 1734 Bishop George Berkeley asked how calculus could divide by a tiny step h and then throw h away, as if it
were zero. He called those vanishing steps "the ghosts of departed quantities". The answer that took hold is the
**limit**: h never has to be zero, only small enough, and "small enough" is a game you can always win.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
Run button) to run it. Run each cell in turn, from the top. When a slider appears, drag it and watch. Change any
number you like and run the cell again. Nothing can break. It works in your browser (no install) and in Google
Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this notebook. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the curve · slopes and windows · areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def slope3(h):
    """Berkeley's working: the slope of y = x² from x = 3 to x = 3 + h."""
    return ((3 + h) ** 2 - 9) / h

print("ready: slope3(0.1) =", round(slope3(0.1), 6))

## 1 · Berkeley's working

Take the curve y = x² and the point (3, 9). Step along by h. The height becomes (3 + h)², so the rise is
(3 + h)² − 9 = 6h + h². Divide by the step h, and the slope of that little step is **6 + h**.

Drag h towards zero. The step line gets closer to the curve's own slope. But try to put h = 0 itself: the slope
becomes 0 ÷ 0, which is no number at all. That was Berkeley's complaint.

In [ ]:
for h in (0.1, 0.01, 0.001):
    print(f"h = {h:<6}  slope = ((3 + h)² − 9) ÷ h = {slope3(h):.6f}")
try:
    slope3(0)
except ZeroDivisionError:
    print("h = 0       slope = 0 ÷ 0: no answer at all")

def step(h=0.8):
    x = np.linspace(0, 5, 300)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, x ** 2, color=TEAL, label="y = x²")
    ax.plot([3, 3 + h], [9, (3 + h) ** 2], "o", color=CHALK, ms=7)
    xs = np.array([1.2, 4.8])
    ax.plot(xs, 9 + slope3(h) * (xs - 3), color=SAGE, lw=2, label="the step's line")
    ax.plot([3, 3 + h, 3 + h], [9, 9, (3 + h) ** 2], color=DIM, lw=1.2, ls=":")
    ax.set_xlim(0, 5); ax.set_ylim(0, 25); ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(loc="upper left")
    ax.set_title(f"h = {h:.2f}:  slope = 6 + h = {slope3(h):.2f}", color=CORAL)
    plt.show()

interact(step, h=FloatSlider(value=0.8, min=0.01, max=1.5, step=0.01, description="step h"));

**Notice.** The slope is always 6 + h, an ordinary number, because h is never zero. It heads to 6 as h shrinks.

## 2 · The epsilon-delta game

Karl Weierstrass turned "small enough" into a game. You name a distance **epsilon** (ε): how close the answer must
be to the limit. I must reply with a distance **delta** (δ): how close x must be to a. I win if every x within δ of
a (but not a itself) gives an answer within ε of the limit.

The figure zooms in on the window. The coral dashed lines are your band, limit ± ε. The green window is mine,
a ± δ. The curve is chalk wherever it escapes the band inside the window.

**Play.** Set ε to 0.01. Then drag δ down until the whole curve inside the window stays in the band.

In [ ]:
GAMES = {
    "y = x², near x = 3 (heads to 9)": (lambda x: x ** 2, 3.0, 9.0),
    "y = 3x + 1, near x = 2 (heads to 7)": (lambda x: 3 * x + 1, 2.0, 7.0),
    "Berkeley's slope 6 + h, near h = 0 (heads to 6)": (slope3, 0.0, 6.0),
}

def game(epsilon=0.1, delta=0.02, curve=list(GAMES)[0]):
    f, a, L = GAMES[curve]
    half = 1.6 * delta                                   # the view: a little wider than my window
    x = np.linspace(a - half, a + half, 2001)
    x = x[np.abs(x - a) > 1e-12]                         # never at a itself
    y = f(x)
    inside = np.abs(x - a) < delta
    miss = inside & (np.abs(y - L) >= epsilon)
    fig, ax = plt.subplots(figsize=(9.5, 5))
    ax.axvspan(a - delta, a + delta, color=SAGE, alpha=0.16, label=f"my window: a ± δ")
    ax.axhline(L + epsilon, color=CORAL, ls="--", lw=2, label=f"your band: L ± ε")
    ax.axhline(L - epsilon, color=CORAL, ls="--", lw=2)
    ax.plot(x, y, color=TEAL, lw=2.6)
    ym = np.where(miss, y, np.nan)
    ax.plot(x, ym, color=CHALK, lw=5, label="escapes the band")
    ax.plot([a], [L], "o", mfc=SLATE, mec=CHALK, ms=9)  # the hole: the game never looks at x = a
    ax.set_xlim(a - half, a + half); ax.set_ylim(L - 3 * epsilon, L + 3 * epsilon)
    ax.set_xlabel("x"); ax.set_ylabel("f(x)")
    ax.ticklabel_format(useOffset=False)
    ax.legend(loc="upper left", fontsize=11)
    if miss.any():
        ax.set_title(f"ε = {epsilon:.2f}, δ = {delta:.4f}: some x in the window escape. Smaller δ.")
    else:
        ax.set_title(f"ε = {epsilon:.2f}, δ = {delta:.4f}: every x in the window lands in the band", color=CORAL)
    plt.show()

interact(game, epsilon=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, readout_format=".2f", description="ε (you)"),
         delta=FloatSlider(value=0.02, min=0.0005, max=0.1, step=0.0005, readout_format=".4f", description="δ (me)"),
         curve=Dropdown(options=list(GAMES), description="curve"));

**Notice.** For y = x² near 3, ε = 0.01 needs δ of about 0.0015. Make ε ten times smaller, and δ shrinks ten times
too. However small you name ε, there is always a δ that works. That is exactly what "the limit is 9" means. Try
the line 3x + 1: there δ = ε ÷ 3 always works, because the line is three times as steep.

## 3 · The examiner's limits

When plugging in gives 0 ÷ 0, that is a sign to look closer, not an answer. Three ways to look closer, each checked
with numbers near the point. The last lines check them with sympy, a Python library for algebra.

In [ ]:
rat = lambda x: (x ** 2 - 4) / (x - 2)             # factor: (x − 2)(x + 2) ÷ (x − 2) = x + 2
s3 = lambda x: math.sin(3 * x) / x                 # 3 · (sin 3x ÷ 3x)
far = lambda x: (3 * x ** 2 - x) / (x ** 2 + 5)    # ÷ x² top and bottom: (3 − 1/x) ÷ (1 + 5/x²)
print("(x² − 4) ÷ (x − 2), near 2:    ", [round(rat(2 + d), 6) for d in (0.1, 0.01, 0.001)], "→ 4")
print("sin 3x ÷ x, near 0:            ", [round(s3(d), 5) for d in (0.1, 0.01, 0.001)], "→ 3")
print("(3x² − x) ÷ (x² + 5), x large: ", [round(far(n), 4) for n in (10, 100, 1000)], "→ 3")

import sympy as sp
X = sp.symbols("x")
print("sympy:", sp.limit((X ** 2 - 4) / (X - 2), X, 2), sp.limit(sp.sin(3 * X) / X, X, 0),
      sp.limit((3 * X ** 2 - X) / (X ** 2 + 5), X, sp.oo))

## 4 · (eˣ − 1) ÷ x heads to 1

Another 0 ÷ 0 at x = 0. Drag x towards zero from either side.

In [ ]:
def exp_limit(x=0.5):
    xs = np.linspace(-1.5, 1.5, 600); xs = xs[np.abs(xs) > 1e-9]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    ax.plot(xs, (np.exp(xs) - 1) / xs, color=TEAL, label="(eˣ − 1) ÷ x")
    ax.plot([0], [1], "o", mfc=SLATE, mec=CHALK, ms=10)
    if abs(x) > 1e-9:
        v = (math.exp(x) - 1) / x
        ax.plot([x], [v], "o", color=CORAL, ms=10)
        ax.set_title(f"x = {x:+.3f}:  (eˣ − 1) ÷ x = {v:.4f}", color=CORAL)
    else:
        ax.set_title("x = 0: 0 ÷ 0, so look nearby")
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(0, 2.2); ax.set_xlabel("x"); ax.legend(loc="upper left")
    plt.show()

interact(exp_limit, x=FloatSlider(value=0.5, min=-1.0, max=1.0, step=0.01, description="x"));

**Your challenge.** What is the limit, as x goes to 0, of (e^(3x) − 1) ÷ x? Hint: make it look like a standard limit. Put your answer
in `my_answer` below and run the cell.

In [ ]:
my_answer = None          # ← your answer, e.g. my_answer = 2
true = sp.limit((sp.exp(3 * X) - 1) / X, X, 0)
if my_answer is None:
    print("Put your answer in my_answer and run this cell again.")
else:
    print("Yes: the limit is", true) if sp.simplify(sp.nsimplify(my_answer) - true) == 0 else print("Not yet. Try writing it as 3 × (e^(3x) − 1) ÷ (3x).")

## 5 · Three areas on a circle

Why does sin x ÷ x head to 1? On a circle of radius one, turn through an angle x, **in radians**. Three shapes sit
inside each other: the small triangle (area sin x ÷ 2), the slice of the circle (area x ÷ 2), and the tall
triangle (area tan x ÷ 2). So sin x < x < tan x.

In [ ]:
def areas(x=0.9):
    t = np.linspace(0, x, 100)
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    c = np.linspace(0, np.pi / 2, 200)
    ax.plot(np.cos(c), np.sin(c), color=DIM, lw=1.5)
    ax.fill([0, 1, 1], [0, 0, math.tan(x)], color=LILAC, alpha=0.18, label=f"tall triangle: {math.tan(x) / 2:.4f}")
    ax.fill(np.r_[0, np.cos(t)], np.r_[0, np.sin(t)], color=LILAC, alpha=0.32, label=f"slice: {x / 2:.4f}")
    ax.fill([0, 1, math.cos(x)], [0, 0, math.sin(x)], color=CORAL, alpha=0.35, label=f"small triangle: {math.sin(x) / 2:.4f}")
    ax.plot([0, 1], [0, math.tan(x)], color=CHALK, lw=1.4)
    ax.set_xlim(-0.05, 1.3); ax.set_ylim(-0.05, 1.6); ax.set_aspect("equal")
    ax.legend(loc="upper left"); ax.set_title(f"x = {x:.2f} radians")
    plt.show()

interact(areas, x=FloatSlider(value=0.9, min=0.05, max=1.2, step=0.05, description="x (radians)"));

**Notice.** The order never changes: small triangle < slice < tall triangle. In degrees the slice's area would not
be x ÷ 2, and the whole argument would pick up a factor of π ÷ 180 ≈ 0.01745.

## 6 · The squeeze: sin x ÷ x between its two bounds

Divide the three areas by sin x ÷ 2 and turn each upside down: **cos x < sin x ÷ x < 1**. Both bounds head to 1 as
x goes to zero, so sin x ÷ x is squeezed to 1 as well. Drag the view in towards zero.

In [ ]:
def squeeze(view=1.5):
    xs = np.linspace(-view, view, 800); xs = xs[np.abs(xs) > 1e-9]
    fig, ax = plt.subplots(figsize=(9.5, 5))
    ax.plot(xs, np.ones_like(xs), color=DIM, lw=2, ls="--", label="1")
    ax.plot(xs, np.cos(xs), color=SAGE, lw=2, label="cos x")
    ax.plot(xs, np.sin(xs) / xs, color=TEAL, lw=3, label="sin x ÷ x")
    ax.plot([0], [1], "o", mfc=SLATE, mec=CORAL, mew=2.5, ms=11)
    lo = math.cos(view)
    ax.set_xlim(-view, view); ax.set_ylim(min(lo, 0.0) - 0.02 if view > 1.2 else lo - (1 - lo) * 0.3, 1 + (1 - lo) * 0.3)
    ax.set_xlabel("x (radians)"); ax.legend(loc="lower center")
    ax.set_title(f"at x = {view:.2f}:  {math.cos(view):.5f} < {math.sin(view) / view:.5f} < 1", color=CORAL)
    plt.show()

interact(squeeze, view=FloatSlider(value=1.5, min=0.05, max=1.5, step=0.05, description="view ±"));

**Notice.** However far you zoom in, sin x ÷ x stays trapped between the two bounds, and the trap closes on 1.

## Sources
Berkeley, *The Analyst* (1734), §§ VII and XXXV (ed. D. R. Wilkins, Trinity College Dublin). MacTutor History of
Mathematics: Berkeley, Maclaurin, Bolzano, Cauchy, Weierstrass. Grabiner (1983), "Who gave you the epsilon?",
American Mathematical Monthly 90(3). CBSE Mathematics syllabus, class XI, Limits and Derivatives. All retrieved
2026-10-24.